# **Block 1 — Imports (only what we need)**

We import the dataset, a train/test split, the random forest model, and the same evaluation tools so your process stays consistent across models.

In [1]:
import numpy as np

from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split

from sklearn.ensemble import RandomForestClassifier

from sklearn.metrics import confusion_matrix, accuracy_score, precision_score, recall_score


# **Block 2 — Inputs (X = features, y = labels)**

Random forests are still supervised learning classifiers in this unit, so nothing changes about the inputs: we need a feature matrix X and a label vector y. We use the same kind of dataset as before so the only thing that changes is the model, not the entire setup, which makes it easier to understand what random forests are actually adding.

In [2]:
data = load_breast_cancer()

X = data.data
y = data.target

print("X shape:", X.shape)
print("y shape:", y.shape)
print("First 10 labels:", y[:10])

unique, counts = np.unique(y, return_counts=True)
print("Class counts:", dict(zip(unique, counts)))


X shape: (569, 30)
y shape: (569,)
First 10 labels: [0 0 0 0 0 0 0 0 0 0]
Class counts: {np.int64(0): np.int64(212), np.int64(1): np.int64(357)}


# **Block 3 — Train/Test split**

Just like with trees, we split the data so we can evaluate generalization. A forest can also overfit if you let it, but it usually reduces that risk compared to a single deep tree, and you’ll see that most clearly when you compare train vs test performance.

In [3]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.25,
    random_state=42,
    stratify=y
)

print("Train rows:", X_train.shape[0])
print("Test rows :", X_test.shape[0])


Train rows: 426
Test rows : 143


# **Block 4 — Train a baseline random forest (keep it simple)**

A random forest is many decision trees trained with randomness, then combined. You do not need to implement the randomness yourself; scikit-learn handles it. We keep the settings simple: choose a reasonable number of trees and fix the random seed so results are reproducible.

In [4]:
# Create a random forest model
# n_estimators = number of trees in the forest
# random_state makes results reproducible
rf = RandomForestClassifier(n_estimators=200, random_state=42)

# Train the forest
rf.fit(X_train, y_train)

print("Random forest trained.")


Random forest trained.


# **Block 5 — Predict + Evaluate (confusion matrix + metrics)**

This block evaluates the forest on the test set using the same evaluation tools you used for logistic regression, kNN, and decision trees. The goal is to build the habit that no matter what model you use, you always come back to the same evaluation questions: what types of mistakes are happening and how good is the model according to the metrics that matter.

In [5]:
y_pred = rf.predict(X_test)

cm = confusion_matrix(y_test, y_pred)
print("Confusion matrix:\n", cm)

acc = accuracy_score(y_test, y_pred)
prec = precision_score(y_test, y_pred, zero_division=0)
rec = recall_score(y_test, y_pred)

print("Accuracy :", round(acc, 3))
print("Precision:", round(prec, 3))
print("Recall   :", round(rec, 3))


Confusion matrix:
 [[49  4]
 [ 2 88]]
Accuracy : 0.958
Precision: 0.957
Recall   : 0.978


# **Block 6 — Train vs test accuracy (spot variance reduction)**

This block is the cleanest way to connect random forests to the bias-variance lesson. Single decision trees often show a big gap between training and testing, because they can memorize. A forest often shrinks that gap because averaging across many trees reduces variance and makes predictions more stable.

In [6]:
# Training accuracy
train_pred = rf.predict(X_train)
train_acc = accuracy_score(y_train, train_pred)

# Test accuracy
test_acc = accuracy_score(y_test, y_pred)

print("Train accuracy:", round(train_acc, 3))
print("Test accuracy :", round(test_acc, 3))
print("Train - Test gap:", round(train_acc - test_acc, 3))


Train accuracy: 1.0
Test accuracy : 0.958
Train - Test gap: 0.042


# **Block 7 — Out-of-bag estimate (built-in validation)**

This block is optional, but it is an easy way to show one of the “cool” random forest ideas without adding complexity. With bootstrap sampling, each tree trains on a slightly different sample, which means some training points are left out for that tree. Out-of-bag scoring uses those left-out points as a built-in validation estimate, which often ends up close to test performance.

In [7]:
# Optional: Out-of-bag (OOB) score
# oob_score=True tells the forest to estimate performance using out-of-bag samples
rf_oob = RandomForestClassifier(n_estimators=200, random_state=42, oob_score=True)

rf_oob.fit(X_train, y_train)

print("OOB score (rough built-in validation):", round(rf_oob.oob_score_, 3))


OOB score (rough built-in validation): 0.962
